# QLoRA Fine-Tuning: Intern-Decision-4B (LoRA v6) on Google Colab (T4 / A100 GPU)

Pre-tokenized, decision-masked training dataset embedded (1,740 clean scenarios).
Inference-aligned prompt format: letter-symbol supervision matching Pipeline compile_row.
LoRA r=16, alpha=32, all 7 projections, lr=1.5e-4, 4 epochs.


## 1. Environment & Dependencies

In [ ]:
!nvidia-smi
!pip install -q transformers datasets peft bitsandbytes accelerate

# Verified prebuilt binary for Colab CUDA 13.0 + Python 3.13 (installs in 5s, zero nvcc build)
!pip install https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.7.0/causal_conv1d-1.7.0%2Bcu13torch2.10cxx11abiTRUE-cp313-cp313-linux_x86_64.whl

# Fast Triton-fused linear attention
!pip install -q flash-linear-attention

import causal_conv1d
print("✓ causal-conv1d ready:", causal_conv1d.__file__)

import fla
print("✓ flash-linear-attention ready:", fla.__file__)


## 2. Ingest Pre-Tokenized Decision-Masked SFT Dataset

In [ ]:
import json, gzip, base64
from datasets import Dataset

DATA_B64 = "..."
raw_json = gzip.decompress(base64.b64decode(DATA_B64.encode("ascii"))).decode("utf-8")
formatted_records = json.loads(raw_json)
print(f"Loaded {len(formatted_records)} pre-tokenized records.")

supervised_counts = [sum(1 for l in r["labels"] if l != -100) for r in formatted_records]
print(f"Avg supervised decision tokens per sample: {sum(supervised_counts)/len(supervised_counts):.1f}")

dataset = Dataset.from_list(formatted_records)
print("HuggingFace Dataset ready.")


## 3. Load QLoRA 4-bit Base Model

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "internlm/Intern-Decision-4B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="sdpa",
)

model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()


## 4. Train QLoRA Adapter (Strict Masked Loss)

In [ ]:
import os
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    pad_to_multiple_of=8,
    return_tensors="pt",
    padding=True
)

output_dir = "output_lora"

# A100 High-Throughput config: batch_size=16 saturates Tensor Cores (fits in ~14GB / 80GB VRAM)
# Slashes total steps from 435 to 109 without recomputing activations
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1.5e-4,
    weight_decay=0.01,
    logging_steps=10,
    num_train_epochs=4,
    warmup_steps=10,
    lr_scheduler_type="cosine",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    save_strategy="no",
    dataloader_num_workers=2,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=collator,
)

# Clean up previous checkpoints
import shutil
if os.path.isdir(output_dir):
    print(f"Cleaning previous checkpoint directory '{output_dir}'...")
    shutil.rmtree(output_dir)
os.makedirs(output_dir, exist_ok=True)
print("Starting fast A100 training on 1,740 masked samples across 4 epochs (109 steps total)...")
trainer.train()

# Explicitly save final adapter directly into output_dir and intern_decision_lora
print(f"Saving final LoRA adapter to {output_dir} and intern_decision_lora...")
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
trainer.model.save_pretrained("intern_decision_lora")
tokenizer.save_pretrained("intern_decision_lora")
print("✓ LoRA adapter saved successfully.")


## 5. Export Adapter Package

In [ ]:
import shutil
from google.colab import files

adapter_path = "intern_decision_lora"
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

shutil.make_archive("intern_decision_lora", "zip", adapter_path)
print("Archived to intern_decision_lora.zip")
files.download("intern_decision_lora.zip")

## 6. Comprehensive 600-Sealed Benchmark on A100 (Zero Regex Grid)
Evaluates LoRA v3 vs Base Model directly on A100 across all 3 sealed splits:
- **multipolicy_sealed_100** (100 scenarios)
- **adversarial_sealed_100** (100 scenarios)
- **sealed_eval_400** (400 scenarios)
Uses fast PyTorch inference on A100 (~0.12s per scenario).

In [ ]:
import json, gzip, base64, os, time
BUNDLE_B64 = "..."

# Extract Pipeline and 3 Sealed Holdout Datasets
raw_bundle = gzip.decompress(base64.b64decode(BUNDLE_B64.encode("ascii"))).decode("utf-8")
bundle_data = json.loads(raw_bundle)
os.makedirs("datasets/sealed", exist_ok=True)

for filename, content in bundle_data.items():
    if filename.endswith(".jsonl"):
        with open(os.path.join("datasets/sealed", filename), "w", encoding="utf-8") as f:
            f.write(content)
    elif filename.endswith(".py"):
        with open(filename, "w", encoding="utf-8") as f:
            f.write(content)
print("Unpacked Pipeline_MultiPolicy.py and 3 sealed holdout benchmarks (600 total).")

# Free training VRAM so evaluation pipeline has clean GPU allocation
import gc
try:
    del model
    del trainer
except:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Initialize pipeline using the newly trained output_lora adapter (LoRA v5)
from Pipeline_MultiPolicy import PassiveListeningPipeline
lora_dir = os.path.abspath("output_lora")
os.environ["LORA_CHECKPOINT_PATH"] = lora_dir
pipe = PassiveListeningPipeline(lora_checkpoint=lora_dir)
assert pipe.lora_attached, f"FATAL: LoRA adapter failed to attach from {lora_dir}! Check if adapter_model.safetensors exists."
print("✓ Active LoRA adapter verified on PassiveListeningPipeline.")

# Ground truth baseline metrics for LoRA v1 from official three_way_comparison_report.json
FROZEN = {
    "multipolicy_100": {"v1_exact": 90.00, "v1_action": 96.00, "v1_dates": 94.00},
    "adversarial_100": {"v1_exact": 78.00, "v1_action": 79.00, "v1_dates": 91.00},
    "baseline_400":    {"v1_exact": 92.50, "v1_action": 93.75, "v1_dates": 94.50},
}

splits = [
    ("multipolicy_100", "datasets/sealed/multipolicy_sealed_100.jsonl"),
    ("adversarial_100", "datasets/sealed/adversarial_sealed_100.jsonl"),
    ("baseline_400",    "datasets/sealed/sealed_eval_400.jsonl"),
]

results = {}
for split_name, path in splits:
    print(f"\n=== EVALUATING LoRA v5: {split_name} ===")
    with open(path, "r", encoding="utf-8") as f:
        cases = [json.loads(line) for line in f if line.strip()]
    
    v5_exact = 0
    v5_dates = 0
    v5_action = 0
    t0 = time.time()
    
    for i, sc in enumerate(cases, 1):
        gold = sc.get("gold_call", {})
        cust = sc.get("student_id", "STD_TEST")
        turns = sc.get("turns", [])
        clock = sc.get("clock")
        
        # 1 fast forward pass per case (LoRA v5 only, zero adapter-switching overhead)
        pred = pipe.process_message_thread(cust, turns, clock)
        act_ok = pred.get("action") == gold.get("action")
        dat_ok = sorted(pred.get("dates") or []) == sorted(gold.get("dates") or [])
        ses_ok = pred.get("session") == gold.get("session")
        
        if act_ok: v5_action += 1
        if dat_ok: v5_dates += 1
        if act_ok and dat_ok and ses_ok: v5_exact += 1
        
        if i % 25 == 0 or i == len(cases):
            print(f"[{i:3d}/{len(cases)}] v5 Exact: {v5_exact/i*100:5.1f}% | Dates: {v5_dates/i*100:5.1f}% | Action: {v5_action/i*100:5.1f}%")
    
    elapsed = time.time() - t0
    results[split_name] = {
        "total": len(cases),
        "v5_exact": v5_exact / len(cases) * 100.0,
        "v5_action": v5_action / len(cases) * 100.0,
        "v5_dates": v5_dates / len(cases) * 100.0,
        "runtime_sec": elapsed,
        "f": FROZEN[split_name]
    }

print("\n" + "="*90)
print("OFFICIAL BENCHMARK: LoRA v1 (1200) vs LoRA v5 (1740 Clean Full Projections) Across 600 Sealed Scenarios")
print("="*90)
print(f"{"Split":<18} | {"LoRA v1 Exact":<14} | {"LoRA v5 Exact":<14} | {"Dates (v1 vs v5)":<18} | {"Delta (v5-v1)":<12}")
print("-"*90)

tot = sum(r["total"] for r in results.values())
weighted_v1 = sum(r["f"]["v1_exact"] * r["total"] for r in results.values()) / tot
weighted_v5 = sum(r["v5_exact"] * r["total"] for r in results.values()) / tot

for name, r in results.items():
    v1_e = r["f"]["v1_exact"]
    v5_e = r["v5_exact"]
    v1_d = r["f"]["v1_dates"]
    v5_d = r["v5_dates"]
    delta = v5_e - v1_e
    d_str = f"+{delta:.2f}%" if delta >= 0 else f"{delta:.2f}%"
    print(f"{name:<18} | {v1_e:5.2f}%        | {v5_e:5.2f}%        | {v1_d:5.2f}% -> {v5_d:5.2f}%  | {d_str}")

delta_all = weighted_v5 - weighted_v1
all_d_str = f"+{delta_all:.2f}%" if delta_all >= 0 else f"{delta_all:.2f}%"
print("-"*90)
print(f"ALL 600 COMBINED   | {weighted_v1:5.2f}%        | {weighted_v5:5.2f}%        | Overall Delta:     | {all_d_str}")
print("="*90)
